# DPAC Clustering

Logan Wong

law3082

Load the embeddings

Do pretraining? NO!!! That was only to learn features from images.

Do clustering using DPAC

In [1]:
import sys
import os

# Add the DPAC program folder to path
dpac_path = '/home/stu5/s5/law3082/Courses/MLDD/Deep-Probability-Aggregation-Clustering/PAC_DPAC_program/'
if dpac_path not in sys.path:
    sys.path.append(dpac_path)

In [2]:
import pandas as pd
import time

import argparse
import torch
import numpy as np
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# from data.contrastive_learning_dataset import ContrastiveLearningDataset

from models import Network  #, get_resnet, get_resnet_cifar, get_resnet_stl
from contrastive_loss import InfonceLoss
# from utils import save_model
import torch.nn.functional as F

In [3]:
os.environ["CUDA_VISIBLE_DEVICES"]="6"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
def convert(seconds):
    seconds = seconds % (24 * 3600)
    hour = seconds // 3600
    seconds %= 3600
    minutes = seconds // 60
    seconds %= 60

    return "%d:%02d:%02d" % (hour, minutes, seconds)

# Load Embeddings

In [5]:
embeddings_raw = np.load('data/event2012_embeddings.npy')
# Convert to a Float Tensor & move to GPU
embeddings = torch.from_numpy(embeddings_raw).float().to(device)

# Load the metadata to track tweet IDs
metadata = pd.read_csv('data/event2012_metadata.csv')

print(f"Loaded {embeddings.shape[0]} embeddings with dimension {embeddings.shape[1]}")

Loaded 68841 embeddings with dimension 768


In [6]:
# class TwitterDataset(Dataset):
#     def __init__(self, embedding_path):
#         # Load your .pt or .npy file here
#         self.embeddings = torch.load(embedding_path) 
#     def __len__(self):
#         return len(self.embeddings)
#     def __getitem__(self, idx):
#         x = self.embeddings[idx]
#         # DPAC's train_model expects: ((weak, strong, ori), label)
#         # We simulate "views" by adding tiny noise to your vectors
#         weak = x + torch.randn_like(x) * 1e-3
#         strong = x + torch.randn_like(x) * 1e-2
#         ori = x        
#         return (weak, strong, ori), 0 # 0 is a dummy label


In [7]:
class TwitterVectorDataset(Dataset):
    def __init__(self, vecs):
        self.vecs = vecs
        
    def __len__(self):
        return len(self.vecs)

    def __getitem__(self, idx):
        x = self.vecs[idx]
        # DPAC expects (weak, strong, ori). 
        # We simulate "views" by adding 1% and 5% noise to the vectors.
        weak = x + torch.randn_like(x) * 0.01
        strong = x + torch.randn_like(x) * 0.05
        return (weak, strong, x), 0 # 0 is a placeholder label

In [8]:
# Initialize loader for training loop
dataset = TwitterVectorDataset(embeddings)
ins_train_loader = DataLoader(dataset, batch_size=256, shuffle=True, drop_last=True)

# a mismatch between your batch size and the number of tweets in your dataset.
# total number of tweets isn't perfectly divisible by 256, the very last batch is smaller (a "partial batch")
# tell DataLoader to ignore the last tiny leftover piece of data
# DataLoader drops the "partial batch" 

In [9]:
# class ClusteringModel(nn.Module):
#     def __init__(self, input_dim, num_clusters):
#         super().__init__()
#         self.encoder = nn.Sequential(
#             nn.Linear(input_dim, 512),
#             nn.ReLU(),
#             nn.Linear(512, 128) # This is your 'z' space
#         )
#         self.cluster_head = nn.Linear(128, num_clusters) # This is your 'p' space

#     def forward(self, x):
#         z = self.encoder(x)
#         p = self.cluster_head(z)
#         return z, p

In [10]:
# Define a Backbone for vectors
class TwitterBackbone(nn.Module):
    def __init__(self, input_dim=768, rep_dim=128):
        super().__init__()
        self.rep_dim = rep_dim # The Network class NEEDS this attribute
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, rep_dim)
        )
        
    def forward(self, x):
        return self.encoder(x)

In [11]:
# input_dim = embeddings.shape[1] # This will be 768 or 1024
# feature_dim = 128               # DPAC's internal 'z' space
# num_events = 50                 # How many events you want to find

# # Create a Backbone that reads vectors instead of pixels
# res = nn.Sequential(
#     nn.Linear(input_dim, 512),
#     nn.ReLU(),
#     nn.Linear(512, feature_dim)
# )
# res.rep_dim = feature_dim # Required attribute for the Network class

# # Use the original DPAC Network class for the clustering logic
# model = Network(res, res.rep_dim, num_events).to('cuda:0')

In [12]:
input_dim = embeddings.shape[1]  # 768 I think
feature_dim = 128                # DPAC's standard 'z' space dimension
num_events = 503                 # Your target number of clusters (events)

my_backbone = TwitterBackbone(input_dim=input_dim, rep_dim=feature_dim)

# Wrap backbone in the DPAC Network class and move to GPU
# Change class_num if you have a specific number of events in mind
model = Network(my_backbone, my_backbone.rep_dim, class_dim=num_events).to(device)

print(f"Model initialized on {device} with input dimension {input_dim}")

Model initialized on cuda with input dimension 768


In [13]:
# Define the optimizer (LR = 0.0001 is standard for DPAC)
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=0.0001)

# Define the Contrastive Loss (InfoNCE)
# batch_size should match your DataLoader (256)
criterion = InfonceLoss(batch_size=256, temperature=0.5, device=device).to(device)

# Setup the GradScaler for mixed-precision training (faster on Narnia's GPUs)
scaler = torch.cuda.amp.GradScaler()

print(f"Optimizer and Criterion ready on {device}")

Optimizer and Criterion ready on cuda


/tmp/ipykernel_37308/1033229026.py:9: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


# Training

In [14]:
def kldiv(q, p):
    """ Standard KL Divergence for clustering """
    res = -torch.sum(q * torch.log(p + 1e-8), dim=-1)
    return res.mean()

In [15]:
def train_model(model, ins_train_loader, optimizer, criterion, scaler, device, m=1.03):
    model.train()
    loss_epoch = {'loss1': 0, 'loss2': 0}
    
    for step, ((weak, strong, ori), _) in enumerate(ins_train_loader):
        # Move all versions of the tweet vector to the GPU
        weak = weak.to(device)
        strong = strong.to(device)
        ori = ori.to(device)
        
        # Concatenate weak and strong for contrastive learning
        img = torch.cat((weak, strong), dim=0)
        
        optimizer.zero_grad()
        
        # Use Mixed Precision for speed on Narnia
        with torch.amp.autocast('cuda'):
            # 1. Forward pass
            z, p1, _ = model(img)
            
            # 2. DPAC Clustering logic (Probability Aggregation)
            # This generates the target distribution 'q'
            q, p = model.PAC_online(ori, m=m) 
            
            # 3. Calculate Losses
            loss1 = criterion(z)         # Contrastive Loss (keeps tweets together)
            loss2 = kldiv(q, p1)         # Clustering Loss (pushes tweets into events)
            
            loss = loss1 + loss2

        # Backpropagation
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        loss_epoch['loss1'] += loss1.item() / len(ins_train_loader)
        loss_epoch['loss2'] += loss2.item() / len(ins_train_loader)
        
    return loss_epoch

In [16]:
# Train and Save the event-clustering model
epochs = 200
model_save_path = 'models/event2012_DPAC_model.tar'

print("Training started")
start_time = time.time()
for epoch in tqdm(range(epochs)):
    # Run one full pass through your Twitter data
    losses = train_model(model, ins_train_loader, optimizer, criterion, scaler, device)
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | "
              f"Contrastive Loss: {losses['loss1']:.4f} | "
              f"Clustering Loss: {losses['loss2']:.4f}")
        
    # Save the model weights
    if (epoch + 1) % 50 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': losses,
        }, f"checkpoints/checkpoint_epoch_{epoch+1}.tar")

end_time = time.time()
print("Training done")

# Calculate run time
training_duration = end_time - start_time
time_in_minutes_and_seconds = convert(training_duration)

Training started


  5%|█████▎                                                                                                     | 10/200 [01:42<32:42, 10.33s/it]

Epoch [10/200] | Contrastive Loss: nan | Clustering Loss: nan


 10%|██████████▋                                                                                                | 20/200 [03:24<30:34, 10.19s/it]

Epoch [20/200] | Contrastive Loss: nan | Clustering Loss: nan


...

Epoch [180/200] | Contrastive Loss: nan | Clustering Loss: nan


 95%|████████████████████████████████████████████████████████████████████████████████████████████████████▋     | 190/200 [32:33<01:46, 10.60s/it]

Epoch [190/200] | Contrastive Loss: nan | Clustering Loss: nan


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [34:19<00:00, 10.30s/it]

Epoch [200/200] | Contrastive Loss: nan | Clustering Loss: nan
Training done


In [17]:
print(f"Time taken: {time_in_minutes_and_seconds}")
# expected to take 33 minutes

Time taken: 0:34:19


In [18]:
# parser = argparse.ArgumentParser()
# parser.add_argument('-dataset-name', default='cifar10',
#                     help='dataset name',
#                     choices=['stl10', 'cifar10', 'cifar100', 'imagenet10', 'imagenet_dogs', 'tiny_imagenet'])
# parser.add_argument('-data', metavar='DIR', default='./datasets',
#                     help='path to dataset')
# parser.add_argument('-model-path', default='./save/CIFAR-10',
#                     help='path to save model')
# # Training Hyper parameter
# parser.add_argument('--epochs', default=200, type=int, metavar='N',
#                     help='number of total epochs to run')
# parser.add_argument('-b', '--batch-size', default=256, type=int,
#                     metavar='N',
#                     help='mini-batch size (default: 240), this is the total '
#                          'batch size of all GPUs on the current node when '
#                          'using Data Parallel or Distributed Data Parallel')
# parser.add_argument('--lr', '--learning-rate', default=1e-4, type=float,
#                     metavar='LR', help='initial learning rate', dest='lr')
# parser.add_argument('--wd', '--weight-decay', default=1e-4, type=float,
#                     metavar='W', help='weight decay (default: 1e-4)',
#                     dest='weight_decay')
# parser.add_argument('--resnet', default='ResNet34', help='Choice resnet.')

# # Hyper parameter
# parser.add_argument('--temperature', default=0.5, type=float, help='softmax temperature (default: 0.5)')
# parser.add_argument('--m', default=1.03, type=float, help='weight exponent > 1 (default: 1.03)')
# parser.add_argument('--thd', default=0.99, type=float, help='threshold of pseudo label (default: 0.95)')

# # Deployment
# parser.add_argument('--gpu-index', default=0, type=int, help='Gpu index.')
# parser.add_argument('--seed', default=0, type=int)


# # NEW!!! I ADDED THIS!!!
# def kldiv(q, p):
#     """
#     Standard KL Divergence for clustering.
#     q: Target distribution (sharpened)
#     p: Predicted distribution
#     """
#     res = -torch.sum(q * torch.log(p + 1e-8), dim=-1)
#     return res.mean()
    

# def pac_loss(p, f):
#     N, C = p.shape
#     p = F.softmax(p, dim=1)
#     dis = 1 - 1 * torch.matmul(f, f.T)
#     ps = torch.mm(p, p.T)
#     loss = (dis * ps).sum(1)
#     return loss.sum() / N


# def train_model(args, ins_train_loader, optimizer, criterion, model, scaler):
#     loss_epoch = {'loss1': 0, 'loss2': 0, 'loss3': 0}
#     for step, ((weak, strong, ori), _) in enumerate(ins_train_loader):
#         weak = weak.to(args.device)
#         strong = strong.to(args.device)
#         img = torch.cat((weak, strong), dim=0)
#         ori = ori.to(args.device)
#         optimizer.zero_grad()
#         with torch.cuda.amp.autocast():
#             z, p1, u2 = model(img)
#             q, p = model.PAC_online(ori, m=args.m)  # clustering codes
#             # loss1 = criterion(z, p)  # contrastive learning
#             loss1 = criterion(z)
#             loss2 = kldiv(q, p1)  # online clustering
#             """ self-labeling fine-tuning same as Fixmatch"""
#             # max_probs, tragets_p = torch.max(F.softmax(p1, dim=1), dim=-1)  # pseudo labels
#             # mask = max_probs.ge(args.thd).float()
#             # loss3 = (F.cross_entropy(u2, tragets_p, reduction='none') * mask).mean()  # self-labeling
#             loss = loss1 + loss2
#         scaler.scale(loss).backward()
#         scaler.step(optimizer)
#         scaler.update()
#         loss_epoch['loss1'] += loss1.item() / len(ins_train_loader)
#         loss_epoch['loss2'] += loss2.item() / len(ins_train_loader)
#         # loss_epoch['loss3'] += loss3.item() / len(ins_train_loader)
#     return model, loss_epoch


# def main():
#     """ DPAC """
#     args = parser.parse_args()
#     args.device = torch.device(f'cuda:{args.gpu_index}')
#     torch.cuda.set_device(args.gpu_index)
#     print(f'select device:cuda{args.gpu_index}')
#     torch.manual_seed(args.seed)
#     torch.cuda.manual_seed_all(args.seed)
#     torch.cuda.manual_seed(args.seed)
#     np.random.seed(args.seed)

#     dataset = ContrastiveLearningDataset(args.data)
#     ins_train_dataset, class_num = dataset.get_dataset(args.dataset_name, train_dataset=True)
#     ins_train_loader = DataLoader(ins_train_dataset, batch_size=args.batch_size, shuffle=True, pin_memory=True,
#                                   num_workers=4, drop_last=True)
#     if args.dataset_name == 'cifar10' or args.dataset_name == 'cifar100':
#         res = get_resnet_cifar(args.resnet)
#     elif args.dataset_name == 'stl10':
#         res = get_resnet_stl(args.resnet)
#     else:
#         res = get_resnet(args.resnet)
#     model = Network(res, res.rep_dim, class_num)
#     model = model.to(args.device)
#     checkpoint = torch.load('./save/CL_1000.tar', map_location=args.device)
#     model.load_state_dict(checkpoint['net'], strict=False)
#     optimizer = torch.optim.Adam(model.parameters(), 1e-4, weight_decay=1e-4)
#     criterion = InfonceLoss(args.batch_size, args.temperature, args.device).to(args.device)
#     torch.backends.cudnn.benchmark = True
#     torch.backends.cuda.matmul.allow_tf32 = True
#     torch.backends.cudnn.allow_tf32 = True
#     scaler = torch.cuda.amp.GradScaler()

#     for epoch_counter in tqdm(range(args.epochs)):
#         model, loss_epoch = train_model(args, ins_train_loader, optimizer, criterion, model, scaler)
#         print(
#             f"Epoch [{epoch_counter}/{args.epochs}]\t "
#             f"loss1_epoch: {loss_epoch['loss1']}\t "
#             f"loss2_epoch: {loss_epoch['loss2']}\t "
#             f"loss3_epoch: {loss_epoch['loss3']}\t "
#         )
#         save_model(args, model, optimizer)


# if __name__ == '__main__':
#     main()
